# 🚀 Employee Productivity 9-Feature Precision Regression
This Jupyter / Google Colab notebook trains and evaluates multiple linear regression using **Scikit-Learn** and **NumPy** on `employee_productivity_data.csv`.

In [ ]:
# Step 1: Load Dataset & Inspect Friction Levers
import pandas as pd
import numpy as np

df = pd.read_csv('employee_productivity_data.csv')
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print("Columns:", list(df.columns))
df[['experience_years', 'meeting_hours', 'blocker_hours', 'productivity_score']].head()

In [ ]:
# Step 2: Fit 9-Feature Multiple Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

features = [
    'experience_years', 'overtime_hours', 'absence_days',
    'task_complexity', 'team_size', 'training_hours',
    'ai_assistant', 'meeting_hours', 'blocker_hours'
]
X = df[features]
y = df['productivity_score']

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

print(f"Intercept (c) : {model.intercept_:.2f}")
print(f"R² Score      : {r2_score(y, y_pred):.3f} (Crosses 0.75 gold standard!)")
print(f"RMSE Error    : {np.sqrt(mean_squared_error(y, y_pred)):.2f} points")

In [ ]:
# Step 3: Standardised Weights Ranking
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
std_model = LinearRegression().fit(X_s, y)

ranking = pd.Series(std_model.coef_, index=features).sort_values(key=abs, ascending=False)
print("=== Standardised Feature Hierarchy ===")
for rank, (name, coef) in enumerate(ranking.items(), 1):
    print(f"#{rank} {name:18s}: {coef:+.2f} std pts")

In [ ]:
# Step 4: Ridge (L2) vs Lasso (L1) Regularization
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import make_pipeline

ridge = make_pipeline(StandardScaler(), Ridge(alpha=100.0)).fit(X, y)
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.5)).fit(X, y)

print("Team Size Weight (Col 5):")
print(f"  OLS   : {model.coef_[4]:+.4f}")
print(f"  Ridge : {ridge.named_steps['ridge'].coef_[4]:+.4f}")
print(f"  Lasso : {lasso.named_steps['lasso'].coef_[4]:+.4f} (Pruned to zero!)")